In [1]:
from pyspark.sql import SparkSession

In [2]:
spark=SparkSession.builder \
    .appName("RDD Operations") \
    .getOrCreate()

In [3]:
customer_data=[
"customer_id, name, city, state, country, registration date, is_active",
"0, Customer 0, Bangalore, Karnataka, India, 2023-11-11, True",
"1, Customer 1, Hyderabad, Delhi, India, 2023-08-26, True",
"2, Customer 2, Ahmedabad, West Bengal, India, 2023-06-23, True",
"3, Customer 3, Bangalore, Tamil Nadu, India, 2023-03-24, False",
"4, Customer 4, Bangalore, Gujarat, India, 2023-06-06, False",
"5, Customer 5, Delhi, Maharashtra, India, 2023-04-19, False",
]

In [4]:
data_rdd=spark.sparkContext.parallelize(customer_data)

In [5]:
data_rdd.getNumPartitions()

2

In [6]:
## RDD are rsilent disrtibuted dataset

In [7]:
# first() - retirn the first elemnet of rdd
header=data_rdd.first()

In [8]:
header

'customer_id, name, city, state, country, registration date, is_active'

In [9]:
# filter ()
data_rdd=data_rdd.filter(lambda row :row !=header)

In [10]:
data_rdd.collect()

['0, Customer 0, Bangalore, Karnataka, India, 2023-11-11, True',
 '1, Customer 1, Hyderabad, Delhi, India, 2023-08-26, True',
 '2, Customer 2, Ahmedabad, West Bengal, India, 2023-06-23, True',
 '3, Customer 3, Bangalore, Tamil Nadu, India, 2023-03-24, False',
 '4, Customer 4, Bangalore, Gujarat, India, 2023-06-06, False',
 '5, Customer 5, Delhi, Maharashtra, India, 2023-04-19, False']

In [11]:
def parse_row(row):
  fields=row.split(",")
  return (
      int(fields[0]),
      fields[1].strip(),
      fields[2].strip(),
      fields[3].strip(),
      fields[4].strip(),
      fields[5].strip(),
      fields[6].strip()=='True'
  )


In [12]:
# Map()
parsed_rdd=data_rdd.map(parse_row)

In [13]:
parsed_rdd.collect()

[(0, 'Customer 0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer 1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer 2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer 3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer 4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer 5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

In [14]:
#ADv Analytics  RDD op


In [15]:
#Extract the field with Map() - i want City And Customer
name_city_rdd=parsed_rdd.map(lambda row:(row[1],row[2]))


In [16]:
name_city_rdd.collect()

[('Customer 0', 'Bangalore'),
 ('Customer 1', 'Hyderabad'),
 ('Customer 2', 'Ahmedabad'),
 ('Customer 3', 'Bangalore'),
 ('Customer 4', 'Bangalore'),
 ('Customer 5', 'Delhi')]

In [17]:
#Filter out the active customer
active_customers=parsed_rdd.filter(lambda row :row[6]==True)

In [18]:
active_customers.collect()

[(0, 'Customer 0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer 1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer 2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

In [19]:
cities_rdd=parsed_rdd.map(lambda row : row[2]).distinct()

In [20]:
cities_rdd.collect()

['Hyderabad', 'Delhi', 'Bangalore', 'Ahmedabad']

In [21]:
# take
cities_rdd.take(2)

['Hyderabad', 'Delhi']

In [22]:
# Reduced By Key
# -- Combines values for each key using an associative reduce function



In [23]:
customers_per_city=parsed_rdd.map(lambda row:(row[2],1)).reduceByKey(lambda x,y:x+y)

In [24]:
customers_per_city

PythonRDD[16] at RDD at PythonRDD.scala:56

In [25]:
customers_per_city.collect()

[('Hyderabad', 1), ('Delhi', 1), ('Bangalore', 3), ('Ahmedabad', 1)]

In [26]:
# CountByVlaue

In [27]:
cust_per_city=parsed_rdd.map(lambda row:row[2]).countByValue()

In [28]:
cust_per_city

defaultdict(int, {'Bangalore': 3, 'Hyderabad': 1, 'Ahmedabad': 1, 'Delhi': 1})

In [29]:
#More Op

In [30]:
#city with active customers
active_cities=parsed_rdd.filter(lambda row: row[6]==True) \
                        .map(lambda row : (row[1],row[2]))

In [31]:
active_cities.collect()

[('Customer 0', 'Bangalore'),
 ('Customer 1', 'Hyderabad'),
 ('Customer 2', 'Ahmedabad')]

In [32]:
#count active customer by state
active_state=parsed_rdd.filter(lambda row : row[6]==True) \
                        .map(lambda row : (row[1],row[3]))

In [33]:
active_state.collect()


[('Customer 0', 'Karnataka'),
 ('Customer 1', 'Delhi'),
 ('Customer 2', 'West Bengal')]

In [34]:
spark.stop()